In [5]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")

# === Clean and Prepare ===
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])

merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ID ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

# === Flag for SJS cases ===
def has_sjs(ae_list):
    return any("stevens-johnson syndrome" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies for AR Calculation ===
counts = defaultdict(int)
total_cases = len(grouped)

for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            counts[(pair, "AB1")] += 1
        counts[(pair, "AB+")] += 1

counts[("++",)] = total_cases

# === Compute AR Metrics ===
results = []

for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nBplus = counts.get((drugB, "B+"), 0)
        nAll = counts[("++",)]

        if nAB1 < 3:
            continue

        support_AC = nA1 / nAll if nAll else 0
        confidence = nAB1 / (nABplus + nBplus) if (nABplus + nBplus) else 0
        lift = confidence / support_AC if support_AC else 0
        conviction = (1 - support_AC) / (1 - confidence) if confidence != 1 else float('inf')

        if lift > 1 and conviction > 1:
            results.append({
                "Drug A": drugA,
                "Drug B": drugB,
                "nAB1": nAB1,
                "Lift": round(lift, 3),
                "Conviction": round(conviction, 3)
            })

# === Export Results ===
ar_results_df = pd.DataFrame(results).sort_values(by="Lift", ascending=False)
ar_results_df.to_csv("association_rule_results_SJS.csv", index=False)

print("Top 10 DDI signals for SJS:")
print(ar_results_df.head(10))


Top 10 DDI signals for SJS:
                                                Drug A  \
358                                         Loratadine   
373                                        Afroqualone   
125  Extract from inflamed skin of rabbits inoculat...   
380                                Fluvoxamine Maleate   
254                      Sulfamethoxazole-trimethoprim   
337           Insulin Aspart (genetically recombinant)   
91                                     Flomoxef sodium   
377                                        Domperidone   
84                             Diltiazem hydrochloride   
364                                      Acetaminophen   

                                          Drug B  nAB1     Lift  Conviction  
358  Thiamine disulfide, B6, and B12 combination     4  434.783       1.061  
373                        General cold medicine     3  333.333       1.071  
125                                   Indapamide     6  317.460       1.080  
380                  

In [6]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])
merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

def has_sjs(ae_list):
    return any("stevens-johnson syndrome" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies for CRR ===
counts = defaultdict(int)
total_cases = len(grouped)

for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            counts[(pair, "AB1")] += 1
        counts[(pair, "AB+")] += 1

counts[("++",)] = total_cases

# === Get CRR-based True Interactions ===
true_pairs = set()

for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nB1 = counts.get((drugB, "A1"), 0)
        nAplus = counts.get((drugA, "B+"), 0)
        nBplus = counts.get((drugB, "B+"), 0)

        if nAB1 < 3:
            continue

        # CRR calculation
        p_AB = nAB1 / nABplus if nABplus else 0
        p_A = nA1 / nAplus if nAplus else 0
        p_B = nB1 / nBplus if nBplus else 0
        denom = max(p_A, p_B)
        crr = p_AB / denom if denom else 0

        if crr > 1.5:
            true_pairs.add((drugA, drugB))

            
#true_pairs.to_csv("true_pairs.csv", index=False)
# === Load AR-based Predictions ===
ar_df = pd.read_csv("association_rule_results_SJS.csv")
predicted_pairs = set(tuple(sorted([row["Drug A"], row["Drug B"]])) for _, row in ar_df.iterrows())

# === Compare: Confusion Matrix Elements ===
tp = len(predicted_pairs & true_pairs)
fp = len(predicted_pairs - true_pairs)
fn = len(true_pairs - predicted_pairs)

# === Optional: Metrics ===
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# === Output Results ===
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"F1 Score: {f1:.3f}")


True Positives (TP): 331
False Positives (FP): 52
False Negatives (FN): 0
Precision: 0.864
Recall (Sensitivity): 1.000
F1 Score: 0.927


In [7]:
# Required Libraries
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")

# === Clean and Prepare ===
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])

merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ID ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

def has_sjs(ae_list):
    return any("stevens-johnson syndrome" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies Using FP-Growth Logic ===
counts = defaultdict(int)
total_cases = len(grouped)

# === First Pass: Count Single Drug Frequencies ===
for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

# === Second Pass: Generate Candidate Pairs from Frequent 1-Items ===
freq_items = {drug for (drug, key) in counts if key == "B+" and counts[(drug, "B+")] >= 3}

pair_counts = defaultdict(int)
for _, row in grouped.iterrows():
    drugs = [d for d in set(row["drug"]) if d in freq_items]
    has_sjs = row["has_sjs"]

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            pair_counts[(pair, "AB1")] += 1
        pair_counts[(pair, "AB+")] += 1

counts.update(pair_counts)
counts[("++",)] = total_cases

# === Compute Metrics for Pairs ===
results = []
for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nBplus = counts.get((drugB, "B+"), 0)
        nAll = counts[("++",)]

        if nAB1 < 3:
            continue

        support_AC = nA1 / nAll if nAll else 0
        confidence = nAB1 / (nABplus + nBplus) if (nABplus + nBplus) else 0
        lift = confidence / support_AC if support_AC else 0
        conviction = (1 - support_AC) / (1 - confidence) if confidence != 1 else float('inf')

        if lift > 1 and conviction > 1:
            results.append({
                "Drug A": drugA,
                "Drug B": drugB,
                "nAB1": nAB1,
                "Lift": round(lift, 3),
                "Conviction": round(conviction, 3)
            })

# === Export Results ===
ar_results_df = pd.DataFrame(results).sort_values(by="Lift", ascending=False)
ar_results_df.to_csv("fpgrowth_manual_results_SJS.csv", index=False)

print("Top 10 DDI signals (Manual FP-Growth style) for SJS:")
print(ar_results_df.head(10))


Top 10 DDI signals (Manual FP-Growth style) for SJS:
                                                Drug A  \
358                                         Loratadine   
373                                        Afroqualone   
125  Extract from inflamed skin of rabbits inoculat...   
380                                Fluvoxamine Maleate   
254                      Sulfamethoxazole-trimethoprim   
337           Insulin Aspart (genetically recombinant)   
91                                     Flomoxef sodium   
377                                        Domperidone   
84                             Diltiazem hydrochloride   
364                                      Acetaminophen   

                                          Drug B  nAB1     Lift  Conviction  
358  Thiamine disulfide, B6, and B12 combination     4  434.783       1.061  
373                        General cold medicine     3  333.333       1.071  
125                                   Indapamide     6  317.460       1.08

In [8]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])
merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

def has_sjs(ae_list):
    return any("stevens-johnson syndrome" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies for CRR ===
counts = defaultdict(int)
total_cases = len(grouped)

for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            counts[(pair, "AB1")] += 1
        counts[(pair, "AB+")] += 1

counts[("++",)] = total_cases

# === Get CRR-based True Interactions ===
true_pairs = set()

for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nB1 = counts.get((drugB, "A1"), 0)
        nAplus = counts.get((drugA, "B+"), 0)
        nBplus = counts.get((drugB, "B+"), 0)

        if nAB1 < 3:
            continue

        # CRR calculation
        p_AB = nAB1 / nABplus if nABplus else 0
        p_A = nA1 / nAplus if nAplus else 0
        p_B = nB1 / nBplus if nBplus else 0
        denom = max(p_A, p_B)
        crr = p_AB / denom if denom else 0

        if crr > 1:
            true_pairs.add((drugA, drugB))

            
#true_pairs.to_csv("true_pairs.csv", index=False)
# === Load AR-based Predictions ===
ar_df = pd.read_csv("fpgrowth_manual_results_SJS.csv")
predicted_pairs = set(tuple(sorted([row["Drug A"], row["Drug B"]])) for _, row in ar_df.iterrows())

# === Compare: Confusion Matrix Elements ===
tp = len(predicted_pairs & true_pairs)
fp = len(predicted_pairs - true_pairs)
fn = len(true_pairs - predicted_pairs)

# === Optional: Metrics ===
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# === Output Results ===
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"F1 Score: {f1:.3f}")


True Positives (TP): 356
False Positives (FP): 27
False Negatives (FN): 0
Precision: 0.930
Recall (Sensitivity): 1.000
F1 Score: 0.963


In [9]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")

# === Clean and Prepare ===
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])

merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ID ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

# === Flag for SJS cases ===
def has_sjs(ae_list):
    return any("diarrhea" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies for AR Calculation ===
counts = defaultdict(int)
total_cases = len(grouped)

for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            counts[(pair, "AB1")] += 1
        counts[(pair, "AB+")] += 1

counts[("++",)] = total_cases

# === Compute AR Metrics ===
results = []

for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nBplus = counts.get((drugB, "B+"), 0)
        nAll = counts[("++",)]

        if nAB1 < 3:
            continue

        support_AC = nA1 / nAll if nAll else 0
        confidence = nAB1 / (nABplus + nBplus) if (nABplus + nBplus) else 0
        lift = confidence / support_AC if support_AC else 0
        conviction = (1 - support_AC) / (1 - confidence) if confidence != 1 else float('inf')

        if lift > 1 and conviction > 1:
            results.append({
                "Drug A": drugA,
                "Drug B": drugB,
                "nAB1": nAB1,
                "Lift": round(lift, 3),
                "Conviction": round(conviction, 3)
            })

# === Export Results ===
ar_results_df = pd.DataFrame(results).sort_values(by="Lift", ascending=False)
ar_results_df.to_csv("association_rule_results_diarrhea.csv", index=False)

print("Top 10 DDI signals for diarrhea:")
print(ar_results_df.head(10))


Top 10 DDI signals for diarrhea:
                              Drug A               Drug B  nAB1      Lift  \
28                  Amikacin sulfate    Human haptoglobin     3  1200.000   
20                         Acyclovir    Human haptoglobin     3   703.125   
41                 Human haptoglobin            Melphalan     4   312.500   
29                  Amikacin sulfate            Melphalan     5   291.262   
48         Hydroxyzine hydrochloride  Polymyxin B sulfate     3   290.323   
342                       Didanosine  Nelfinavir Mesylate     3   250.000   
23                         Acyclovir            Melphalan     8   243.902   
44   Hydrocortisone sodium succinate            Melphalan     5   242.718   
47         Hydroxyzine hydrochloride            Melphalan     4   242.424   
102                   Amphotericin B            Melphalan     5   240.385   

     Conviction  
28        1.250  
20        1.230  
41        1.043  
29        1.051  
48        1.051  
342       1

In [10]:
import pandas as pd
from itertools import combinations
from collections import defaultdict

# === Load Data ===
drug_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\drug_table_final.xlsx")
react_df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\DDI\react_table.xlsx")

# === Merge on case ID ===
merged_df = pd.merge(drug_df, react_df, on="Identification number")
merged_df_cleaned = merged_df[[
    "Identification number", 
    "Medicines (generic name)", 
    "Pharmaceutical involvement", 
    "Adverse events"
]].dropna(subset=["Medicines (generic name)", "Adverse events"])
merged_df_cleaned.columns = ["case_id", "drug", "role", "ae"]

# === Group by case ===
grouped = merged_df_cleaned.groupby("case_id").agg({
    "drug": list,
    "ae": list
}).reset_index()

def has_sjs(ae_list):
    return any("diarrhea" in ae.lower() for ae in ae_list)

grouped["has_sjs"] = grouped["ae"].apply(has_sjs)

# === Count Frequencies for CRR ===
counts = defaultdict(int)
total_cases = len(grouped)

for _, row in grouped.iterrows():
    drugs = list(set(row["drug"]))
    has_sjs = row["has_sjs"]

    for drug in drugs:
        if has_sjs:
            counts[(drug, "A1")] += 1
        counts[(drug, "B+")] += 1

    for drugA, drugB in combinations(sorted(drugs), 2):
        pair = (drugA, drugB)
        if has_sjs:
            counts[(pair, "AB1")] += 1
        counts[(pair, "AB+")] += 1

counts[("++",)] = total_cases

# === Get CRR-based True Interactions ===
true_pairs = set()

for key in counts:
    if isinstance(key[0], tuple) and key[1] == "AB1":
        drugA, drugB = key[0]
        nAB1 = counts[((drugA, drugB), "AB1")]
        nABplus = counts.get(((drugA, drugB), "AB+"), 0)
        nA1 = counts.get((drugA, "A1"), 0)
        nB1 = counts.get((drugB, "A1"), 0)
        nAplus = counts.get((drugA, "B+"), 0)
        nBplus = counts.get((drugB, "B+"), 0)

        if nAB1 < 3:
            continue

        # CRR calculation
        p_AB = nAB1 / nABplus if nABplus else 0
        p_A = nA1 / nAplus if nAplus else 0
        p_B = nB1 / nBplus if nBplus else 0
        denom = max(p_A, p_B)
        crr = p_AB / denom if denom else 0

        if crr > 1:
            true_pairs.add((drugA, drugB))

            
#true_pairs.to_csv("true_pairs.csv", index=False)
# === Load AR-based Predictions ===
ar_df = pd.read_csv("association_rule_results_diarrhea.csv")
predicted_pairs = set(tuple(sorted([row["Drug A"], row["Drug B"]])) for _, row in ar_df.iterrows())

# === Compare: Confusion Matrix Elements ===
tp = len(predicted_pairs & true_pairs)
fp = len(predicted_pairs - true_pairs)
fn = len(true_pairs - predicted_pairs)

# === Optional: Metrics ===
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

# === Output Results ===
print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall (Sensitivity): {recall:.3f}")
print(f"F1 Score: {f1:.3f}")


True Positives (TP): 392
False Positives (FP): 33
False Negatives (FN): 0
Precision: 0.922
Recall (Sensitivity): 1.000
F1 Score: 0.960
